In [5]:
#[STUDENT CONFIG]

import time, random

LAST_NAME = "Ganade"
STUDENT_ID = "TUPM-26-3607"
FAVORITE_ARTIST = "Grentperez"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)
ARTIST_LENGTH = len(FAVORITE_ARTIST)
ARTIST_SUM = sum(ord(c) - 64 for c in FAVORITE_ARTIST.upper() if c.isalpha())

print("=== INPUTS ===")
print("SEED_DIGIT:", SEED_DIGIT, "| ID_SUM:", ID_SUM, "| NAME_LENGTH:", NAME_LENGTH)
print("ARTIST_LENGTH:", ARTIST_LENGTH, "| ARTIST_SUM:", ARTIST_SUM)

=== INPUTS ===
SEED_DIGIT: 7 | ID_SUM: 24 | NAME_LENGTH: 6
ARTIST_LENGTH: 10 | ARTIST_SUM: 134


In [6]:
#[EXERCISE 1]
def linear_search(data, target):
    c = 0
    for i, v in enumerate(data):
        c += 1
        if v == target:
            return i, c
    return -1, c

def rotated_binary_search(a, target):
    low, high, c = 0, len(a) - 1, 0
    while low <= high:
        mid = (low + high) // 2
        c += 1
        if a[mid] == target:
            return mid, c
        if a[low] <= a[mid]:                      
            if a[low] <= target < a[mid]:
                high = mid - 1
            else:
                low = mid + 1
        else:                                     
            if a[mid] < target <= a[high]:
                low = mid + 1
            else:
                high = mid - 1
    return -1, c

n = 10 + SEED_DIGIT
original = [100 + ID_SUM + i * NAME_LENGTH for i in range(n)]
k = SEED_DIGIT
rotated = original[k:] + original[:k]
target = original[(ID_SUM + ARTIST_SUM) % n]
missing = target + 1

print("Original:", original)
print("Rotation point k:", k)
print("Rotated :", rotated)
print("Target  :", target)
for label, t in (("Target", target), ("Missing target", missing)):
    bi, bc = rotated_binary_search(rotated, t)
    li, lc = linear_search(rotated, t)
    print(f"{label} {t}: Modified binary -> index={bi}, comparisons={bc} | Linear -> index={li}, comparisons={lc}")

Original: [124, 130, 136, 142, 148, 154, 160, 166, 172, 178, 184, 190, 196, 202, 208, 214, 220]
Rotation point k: 7
Rotated : [166, 172, 178, 184, 190, 196, 202, 208, 214, 220, 124, 130, 136, 142, 148, 154, 160]
Target  : 154
Target 154: Modified binary -> index=15, comparisons=4 | Linear -> index=15, comparisons=16
Missing target 155: Modified binary -> index=-1, comparisons=5 | Linear -> index=-1, comparisons=17


In [7]:
#[EXERCISE 2]

def merge(a, b):
    i = j = comps = 0
    out = []
    while i < len(a) and j < len(b):
        comps += 1
        if a[i] <= b[j]:
            out.append(a[i]); i += 1
        else:
            out.append(b[j]); j += 1
    out.extend(a[i:]); out.extend(b[j:])
    return out, comps

A = [SEED_DIGIT + ID_SUM * i for i in range(1, 9)]
B = [ARTIST_LENGTH + NAME_LENGTH * i for i in range(1, ARTIST_LENGTH + 1)]
merged, comps = merge(A, B)
print("Dept A:", A)
print("Dept B:", B)
print("Consolidated:", merged)
print("Records before:", len(A), "+", len(B), "=", len(A) + len(B), "| after:", len(merged))
print("Comparisons:", comps)
print("Ordering verified:", merged == sorted(A + B))

Dept A: [31, 55, 79, 103, 127, 151, 175, 199]
Dept B: [16, 22, 28, 34, 40, 46, 52, 58, 64, 70]
Consolidated: [16, 22, 28, 31, 34, 40, 46, 52, 55, 58, 64, 70, 79, 103, 127, 151, 175, 199]
Records before: 8 + 10 = 18 | after: 18
Comparisons: 12
Ordering verified: True


In [8]:
#[EXERCISE 3]

def bubble(data):
    a, c, s = data.copy(), 0, 0
    for i in range(len(a)):
        sw = False
        for j in range(len(a) - i - 1):
            c += 1
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]; s += 1; sw = True
        if not sw:
            break
    return a, c, s

def selection(data):
    a, c, s = data.copy(), 0, 0
    for i in range(len(a) - 1):
        m = i
        for j in range(i + 1, len(a)):
            c += 1
            if a[j] < a[m]:
                m = j
        if m != i:
            a[i], a[m] = a[m], a[i]; s += 1
    return a, c, s

def insertion(data):
    a, c, s = data.copy(), 0, 0
    for i in range(1, len(a)):
        key, j = a[i], i - 1
        while j >= 0:
            c += 1
            if a[j] <= key:
                break
            a[j + 1] = a[j]; s += 1; j -= 1
        a[j + 1] = key
    return a, c, s

size = 20 + ID_SUM
rnd = random.Random(ID_SUM + ARTIST_SUM)
base = rnd.sample(range(1, 1000), size)
orders = {
    "Sorted": sorted(base),
    "Reverse": sorted(base, reverse=True),
    "Random": base,
}
print("Dataset size:", size)
print("Random order:", base)

def timeit(f, d, trials=200):
    t = time.perf_counter()
    for _ in range(trials):
        f(d)
    return (time.perf_counter() - t) / trials

algos = [("Bubble Sort", bubble), ("Selection Sort", selection), ("Insertion Sort", insertion)]
print(f"\n{'Order':<9}{'Algorithm':<16}{'Comparisons':>12}{'Movements':>11}{'Time (s)':>14}{'Correct':>9}")
for oname, d in orders.items():
    for aname, f in algos:
        res, c, m = f(d)
        t = timeit(f, d)
        print(f"{oname:<9}{aname:<16}{c:>12}{m:>11}{t:>14.8f}{str(res == sorted(d)):>9}")

Dataset size: 44
Random order: [701, 967, 226, 957, 856, 105, 62, 429, 578, 790, 972, 338, 778, 649, 474, 656, 41, 66, 846, 117, 6, 571, 289, 49, 59, 84, 884, 619, 670, 731, 836, 546, 282, 34, 211, 647, 852, 57, 481, 674, 244, 90, 399, 323]

Order    Algorithm        Comparisons  Movements      Time (s)  Correct
Sorted   Bubble Sort               43          0    0.00000215     True
Sorted   Selection Sort           946          0    0.00006726     True
Sorted   Insertion Sort            43          0    0.00000500     True
Reverse  Bubble Sort              946        946    0.00017688     True
Reverse  Selection Sort           946         22    0.00006036     True
Reverse  Insertion Sort           946        946    0.00013658     True
Random   Bubble Sort              901        545    0.00032894     True
Random   Selection Sort           946         39    0.00015445     True
Random   Insertion Sort           583        545    0.00010210     True
